# 03 · PD model, risk bands and cut-off strategy
The models are trained by `python -m src.pipeline` (see `src/modeling.py`). This notebook reads the results.

**Questions:** how well does the model rank and calibrate risk, which variables drive it, and what approval cut-off gives an acceptable default rate?

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import duckdb
import pandas as pd
GOLD, SILVER, REPORTS = ROOT / "data/gold", ROOT / "data/silver", ROOT / "reports/data_studio/data"
def q(sql: str) -> pd.DataFrame:
    return duckdb.sql(sql.replace("{gold}", GOLD.as_posix()).replace("{silver}", SILVER.as_posix())).df()

## Metrics (validation and test)

In [ ]:
pd.read_csv(REPORTS / "model_metrics.csv").round(4)

## Deciles: calibration and lift (test set)

In [ ]:
pd.read_csv(REPORTS / "model_deciles.csv").round(4)

## Risk bands

In [ ]:
seg = pd.read_csv(REPORTS / "risk_segments.csv")
b = seg.groupby("risk_band")[["n_clients", "n_defaults", "sum_pd"]].sum()
b["default_rate"], b["avg_pd"] = b.n_defaults / b.n_clients, b.sum_pd / b.n_clients
b[["n_clients", "default_rate", "avg_pd"]]

## Cut-off strategy (estimated on the test set)

In [ ]:
cut = pd.read_csv(REPORTS / "cutoff_strategy.csv")
cut[cut.approval_rate_pct.isin([60, 70, 80, 90, 100])].round(4)

## Drivers (permutation importance)

In [ ]:
pd.read_csv(REPORTS / "feature_importance.csv").round(4)

## Notes
_Summarise the model choice, its test metrics, calibration quality and a recommended cut-off, with the trade-off it implies._